In [0]:
# CELL 1: Setup & Load
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, to_date, when, lit

# -------------------------------------------------------------------------
# 1. Azure Storage Config
# -------------------------------------------------------------------------

storage_account_name = "REMOVED_FOR_SECURITY"

storage_account_key = "REMOVED_FOR_SECURITY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Auto-create silver container if missing
spark.conf.set(
    "fs.azure.createRemoteFileSystemDuringInitialization",
    "true"
)

# Optional but useful
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

# -------------------------------------------------------------------------
# 2. Define Paths
# -------------------------------------------------------------------------

base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"

# -------------------------------------------------------------------------
# 3. Read Bronze CSVs
# -------------------------------------------------------------------------

print("Reading Bronze files...")

df_trans_bronze = spark.read.option("header", "true").csv(
    f"{base_path}/transactions/landing/current.csv"
)

df_proj_bronze = spark.read.option("header", "true").csv(
    f"{base_path}/projects/landing/current.csv"
)

df_units_bronze = spark.read.option("header", "true").csv(
    f"{base_path}/units/landing/current.csv"
)

print("✅ Bronze Data Loaded & Silver Container Configured.")

In [0]:
print(df_proj_bronze.columns)

In [0]:
# CELL 2: Clean Projects Dimension
# -------------------------------------------------------------------------

from pyspark.sql.functions import regexp_replace

df_projects_clean = df_proj_bronze.select(

    # FIX malformed values before casting
    regexp_replace(
        col("project_number"),
        "null",
        ""
    ).cast("int").alias("project_number"),

    col("project_status"),

    regexp_replace(
        col("percent_completed"),
        "null",
        ""
    ).try_cast("double").alias("percent_completed"),

    to_date(
        col("project_start_date"),
        "dd-MM-yyyy"
    ).alias("start_date"),

    to_date(
        col("completion_date"),
        "dd-MM-yyyy"
    ).alias("completion_date"),

    col("master_project_en").alias("master_community")

).dropDuplicates(["project_number"])

print("✅ Projects Cleaned.")

display(df_projects_clean.limit(5))

In [0]:
# CELL 3: Clean Transactions Fact
# -------------------------------------------------------------------------

df_trans_clean = df_trans_bronze.select(

    col("transaction_id"),

    # Date
    to_date(
        col("instance_date"),
        "dd-MM-yyyy"
    ).alias("transaction_date"),

    # Transaction Info
    col("procedure_name_en").alias("transaction_type"),

    col("trans_group_en").alias("transaction_group"),

    # Location & Project Info
    col("area_name_en").alias("area_name"),

    col("building_name_en").alias("building_name"),

    col("project_number").try_cast("int"),

    col("project_name_en").alias("project_name"),

    col("nearest_landmark_en").alias("landmark"),

    # Property Details
    col("property_type_en").alias("property_type"),

    col("property_sub_type_en").alias("property_subtype"),

    col("rooms_en").alias("room_type"),

    # Metrics (SAFE CASTS)
    col("procedure_area").try_cast("double").alias("area_sqft"),

    col("actual_worth").try_cast("double").alias("price"),

    col("meter_sale_price").try_cast("double").alias("meter_sale_price")

)

# Remove invalid prices
df_trans_clean = df_trans_clean.filter(
    col("price") > 0
)

print("✅ Transactions Cleaned.")

display(df_trans_clean.limit(5))

In [0]:
display(
    df_trans_clean.select(
        "price",
        "area_sqft",
        "meter_sale_price"
    ).limit(10)
)

In [0]:
# CELL 4: Join & Enrich (Silver Master Table)
# -------------------------------------------------------------------------

# 1. Join Transactions with Projects
df_enriched = df_trans_clean.join(
    df_projects_clean,
    on="project_number",
    how="left"
)

# 2. Calculate Price Per SqFt
df_enriched = df_enriched.withColumn(

    "price_per_sqft",

    when(
        col("area_sqft") > 0,
        col("price") / col("area_sqft")
    ).otherwise(0)

)

# 3. Create Offplan Flag
df_enriched = df_enriched.withColumn(

    "is_offplan_flag",

    when(
        (
            col("transaction_date") < col("completion_date")
        ) |
        (
            col("percent_completed") < 100
        ),
        lit(True)
    ).otherwise(lit(False))

)

print("✅ Data Enriched & Joined.")

display(
    df_enriched.select(
        "transaction_date",
        "project_name",
        "room_type",
        "price",
        "price_per_sqft",
        "is_offplan_flag"
    ).limit(10)
)

In [0]:
display(
    df_enriched.filter(
        col("project_name").isNotNull()
    ).select(
        "project_number",
        "project_name",
        "room_type",
        "price",
        "project_status",
        "percent_completed"
    ).limit(20)
)

In [0]:
display(
    df_trans_clean.filter(
        col("room_type").isNotNull()
    ).select(
        "transaction_type",
        "room_type",
        "project_name",
        "price"
    ).limit(20)
)

In [0]:
# CELL 5: Write to Silver Layer
# -------------------------------------------------------------------------

# Write Transactions Silver
print("Writing Transactions to Silver...")

df_enriched.write.format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/transactions")

# -------------------------------------------------------------------------
# Clean Units
# -------------------------------------------------------------------------

print("Cleaning Units...")

df_units_clean = df_units_bronze.select(

    col("property_id"),

    col("project_id").try_cast("int").alias("project_number"),

    col("unit_number"),

    col("rooms").alias("bedrooms"),

    col("rooms_en").alias("room_type"),

    col("actual_area").try_cast("double").alias("unit_area")

).dropDuplicates(["property_id"])

# -------------------------------------------------------------------------
# Write Units Silver
# -------------------------------------------------------------------------

print("Writing Units to Silver...")

df_units_clean.write.format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/units")

print("🎉 SUCCESS: Silver Files Written.")

In [0]:
# CELL 6: Verify Silver Layer
# -------------------------------------------------------------------------

print("🔍 Verifying Silver Layer...")

try:

    # Read Silver Delta Tables
    df_silver_trans = spark.read.format("delta").load(
        f"{silver_path}/transactions"
    )

    df_silver_units = spark.read.format("delta").load(
        f"{silver_path}/units"
    )

    # Counts
    count_trans = df_silver_trans.count()

    count_units = df_silver_units.count()

    print(f"✅ VERIFIED: Transactions Table = {count_trans} rows")

    print(f"✅ VERIFIED: Units Table = {count_units} rows")

    print("🚀 Preview of Silver Transactions:")

    display(
        df_silver_trans.select(
            "transaction_date",
            "project_name",
            "price",
            "price_per_sqft",
            "is_offplan_flag"
        ).limit(10)
    )

except Exception as e:

    print(f"❌ ERROR: {e}")